# Analytical Placement v2 — grid-spread init + density penalty

`train.ipynb` had a wirelength collapse problem: macros all clustered into one
corner because nothing in the loss said *spread out*. This notebook fixes that
with two changes:

1. **Grid-spread initialization** — movable macros start uniformly across the
   canvas, sorted by area (large macros take corners first).
2. **Density spreading penalty** — a new differentiable loss term that
   penalizes any bin holding more macro area than the canvas average.

Loss = smooth HPWL + ReLU overlap + canvas hinge + **density spread**

No multi-restart in this version. Single, focused training run.

## 1. Setup

In [ ]:
import os
import sys
import math
from pathlib import Path

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR
while not (REPO_ROOT / "macro_place").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

ANALYTICAL_DIR = REPO_ROOT / "submissions" / "analytical"
CHECKPOINT_DIR = ANALYTICAL_DIR / "checkpoints_v2"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

import torch

def select_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = select_device()
if DEVICE.type == "cpu":
    torch.set_num_threads(os.cpu_count() or 1)

print(f"Repo root: {REPO_ROOT}")
print(f"Device:    {DEVICE}")

## 2. Imports

In [ ]:
import time

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display as ipy_display

from macro_place.loader import load_benchmark_from_dir
from submissions.analytical import losses as L
from submissions.core.eval import evaluate as tilos_evaluate

## 3. Load benchmark

In [ ]:
BENCHMARK_NAME = "ibm01"
BENCHMARK_DIR = REPO_ROOT / "external" / "MacroPlacement" / "Testcases" / "ICCAD04" / BENCHMARK_NAME

benchmark, plc = load_benchmark_from_dir(str(BENCHMARK_DIR))
print(benchmark)

# Quick sanity numbers to inform hyperparameter choices.
canvas_area = benchmark.canvas_width * benchmark.canvas_height
macro_area  = float((benchmark.macro_sizes[:benchmark.num_hard_macros, 0]
                   * benchmark.macro_sizes[:benchmark.num_hard_macros, 1]).sum())
print(f"Canvas area:    {canvas_area:.1f} μm²")
print(f"Hard macro area:{macro_area:.1f} μm²  ({100*macro_area/canvas_area:.1f}% utilization)")

## 4. Grid-spread initialization

Place movable hard macros on a uniform grid covering the canvas. Larger
macros are placed first so they claim spots without conflict. The
benchmark's fixed macros stay where they are.

In [ ]:
def grid_spread_init(benchmark, jitter=0.0):
    """Spread movable hard macros uniformly across the canvas."""
    num_hard = benchmark.num_hard_macros
    positions = benchmark.macro_positions.clone().float()
    sizes = benchmark.macro_sizes.float()

    movable = [i for i in range(num_hard) if not bool(benchmark.macro_fixed[i])]
    # Largest first so big macros pick corners and dominate cell choice.
    movable.sort(key=lambda i: -float(sizes[i, 0] * sizes[i, 1]))

    n_movable = len(movable)
    cols = int(math.ceil(math.sqrt(n_movable)))
    rows = (n_movable + cols - 1) // cols
    cell_w = benchmark.canvas_width  / cols
    cell_h = benchmark.canvas_height / rows

    torch.manual_seed(0)
    for k, i in enumerate(movable):
        col = k % cols
        row = k // cols
        cx = (col + 0.5) * cell_w
        cy = (row + 0.5) * cell_h
        if jitter > 0:
            cx += (torch.rand(1).item() - 0.5) * cell_w * jitter
            cy += (torch.rand(1).item() - 0.5) * cell_h * jitter
        positions[i, 0] = cx
        positions[i, 1] = cy

    # Clamp each macro to fit inside canvas.
    half = sizes / 2.0
    positions[:num_hard, 0].clamp_(
        half[:num_hard, 0], float(benchmark.canvas_width)  - half[:num_hard, 0]
    )
    positions[:num_hard, 1].clamp_(
        half[:num_hard, 1], float(benchmark.canvas_height) - half[:num_hard, 1]
    )
    return positions


init_positions = grid_spread_init(benchmark, jitter=0.05)
print(f"Initialized {benchmark.num_hard_macros} hard macros on a grid.")

## 5. Visualization helpers

In [ ]:
def plot_placement(ax, positions, benchmark, title=""):
    ax.clear()
    ax.set_xlim(0, benchmark.canvas_width)
    ax.set_ylim(0, benchmark.canvas_height)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.set_xlabel("x (μm)")
    ax.set_ylabel("y (μm)")

    pos = positions.detach().cpu().numpy()
    sizes = benchmark.macro_sizes.cpu().numpy()
    fixed = benchmark.macro_fixed.cpu().numpy()
    for i in range(benchmark.num_hard_macros):
        x = pos[i, 0] - sizes[i, 0] / 2.0
        y = pos[i, 1] - sizes[i, 1] / 2.0
        color = "lightgray" if fixed[i] else "steelblue"
        ax.add_patch(mpatches.Rectangle(
            (x, y), sizes[i, 0], sizes[i, 1],
            facecolor=color, edgecolor="black",
            alpha=0.55, linewidth=0.4,
        ))


def plot_loss(ax, history):
    ax.clear()
    if not history["step"]:
        return
    ax.plot(history["step"], history["wl"],      label="smooth WL")
    ax.plot(history["step"], history["overlap"], label="overlap")
    ax.plot(history["step"], history["density"], label="density")
    ax.plot(history["step"], history["total"],   label="total", linewidth=2)
    ax.set_yscale("log")
    ax.set_xlabel("step")
    ax.set_ylabel("loss (log)")
    ax.set_title("Smooth proxy loss")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")


# Show the grid-spread starting placement.
fig, ax = plt.subplots(1, 1, figsize=(7, 7))
plot_placement(ax, init_positions, benchmark, title=f"{benchmark.name} — grid-spread init")
plt.tight_layout(); plt.show()

## 6. Training with density penalty

Key change vs `train.ipynb`: a `density_penalty` term is added that
actively pushes macros apart when any bin gets crowded. This balances
the wirelength gradient and prevents collapse.

In [ ]:
def train_v2(
    benchmark,
    plc,
    init_positions,
    *,
    n_epochs=80,
    steps_per_epoch=100,
    lr=3e-3,
    gamma_start=0.01,
    gamma_end=0.002,
    w_overlap=80.0,
    w_canvas=200.0,
    w_density=1500.0,        # NEW — controls spreading strength
    n_bins=8,                # density grid resolution
    checkpoint_dir=CHECKPOINT_DIR,
    device=DEVICE,
    seed=42,
):
    """Single training run with grid-spread init and density penalty."""
    torch.manual_seed(seed)
    total_steps = n_epochs * steps_per_epoch

    positions = init_positions.clone().to(device, dtype=torch.float32).requires_grad_(True)
    sizes_dev = benchmark.macro_sizes.to(device, dtype=torch.float32)
    fixed_dev = benchmark.macro_fixed.to(device)
    anchor    = benchmark.macro_positions.to(device, dtype=torch.float32)
    padded_nets, mask, net_weights = L.prepare_net_tensors(benchmark, device)

    # Clamp out-of-bounds net indices (ports / macro-pins).
    n_pos = positions.shape[0]
    out_of_bounds = padded_nets >= n_pos
    padded_nets = padded_nets.clamp(max=n_pos - 1)
    mask = mask & ~out_of_bounds

    optimizer = torch.optim.Adam([positions], lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_steps, eta_min=lr / 100
    )

    history = {"step": [], "wl": [], "overlap": [], "density": [], "canvas": [], "total": []}
    epoch_log = []
    best_positions = positions.detach().cpu().clone()
    best_total = float("inf")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig_handle = ipy_display(fig, display_id=True)
    start_time = time.perf_counter()

    def lerp(s, e, frac):
        return s + (e - s) * max(0.0, min(1.0, frac))

    for step in range(1, total_steps + 1):
        frac = step / total_steps
        gamma = lerp(gamma_start, gamma_end, frac)

        optimizer.zero_grad()

        wl      = L.smooth_hpwl(positions, padded_nets, mask, net_weights, gamma=gamma)
        overlap = L.overlap_penalty(positions, sizes_dev, benchmark.num_hard_macros)
        density = L.density_penalty(
            positions, sizes_dev, benchmark.num_hard_macros,
            benchmark.canvas_width, benchmark.canvas_height, n_bins=n_bins,
        )
        canvas  = L.canvas_penalty(
            positions, sizes_dev, benchmark.canvas_width, benchmark.canvas_height
        )

        loss = wl + w_overlap * overlap + w_density * density + w_canvas * canvas
        loss.backward()

        with torch.no_grad():
            positions.grad[fixed_dev] = 0.0
        optimizer.step()
        scheduler.step()
        with torch.no_grad():
            positions.data[fixed_dev] = anchor[fixed_dev]

        loss_f = float(loss.detach())
        if loss_f < best_total:
            best_total = loss_f
            best_positions = positions.detach().cpu().clone()

        history["step"].append(step)
        history["wl"].append(float(wl.detach()))
        history["overlap"].append(max(float(overlap.detach()), 1e-12))
        history["density"].append(max(float(density.detach()), 1e-12))
        history["canvas"].append(max(float(canvas.detach()), 1e-12))
        history["total"].append(loss_f)

        if step % steps_per_epoch == 0:
            epoch_idx = step // steps_per_epoch
            elapsed   = time.perf_counter() - start_time

            plot_placement(
                axes[0], positions, benchmark,
                title=(
                    f"{benchmark.name}  epoch {epoch_idx}/{n_epochs}  "
                    f"γ={gamma:.4f}"
                ),
            )
            plot_loss(axes[1], history)
            fig.tight_layout()
            fig_handle.update(fig)

            print(
                f"epoch {epoch_idx:>3d}/{n_epochs}  "
                f"total={loss_f:.2f}  "
                f"wl={float(wl.detach()):.2f}  "
                f"density={float(density.detach()):.4f}  "
                f"overlap={float(overlap.detach()):.4f}  "
                f"lr={scheduler.get_last_lr()[0]:.2e}  "
                f"elapsed={elapsed:.1f}s"
            )

            ckpt_path = checkpoint_dir / f"{benchmark.name}_v2_epoch{epoch_idx:03d}.pt"
            torch.save({
                "benchmark": benchmark.name,
                "epoch": epoch_idx,
                "step": step,
                "positions": positions.detach().cpu(),
                "best_positions": best_positions,
                "history": history,
                "config": {
                    "lr": lr, "w_overlap": w_overlap, "w_density": w_density,
                    "w_canvas": w_canvas, "n_bins": n_bins, "seed": seed,
                },
            }, ckpt_path)
            epoch_log.append({
                "epoch": epoch_idx, "step": step,
                "path": str(ckpt_path), "total": loss_f,
                "wl": float(wl.detach()), "density": float(density.detach()),
            })

    plt.close(fig)
    return best_positions, history, epoch_log

## 7. Run training

In [ ]:
positions, history, epoch_log = train_v2(
    benchmark,
    plc,
    init_positions,
    n_epochs=80,
    steps_per_epoch=100,
    w_density=1500.0,
    w_overlap=80.0,
)

print("\nLast 5 epochs:")
for row in epoch_log[-5:]:
    print(f"  epoch {row['epoch']:>3d}  total={row['total']:.2f}  wl={row['wl']:.2f}  density={row['density']:.4f}")

## 8. Legalize and evaluate with TILOS

Optimization minimizes the *smooth* proxy. This cell pushes any
remaining overlaps apart (using the production placer's legalization)
and scores with the real TILOS evaluator.

In [ ]:
from submissions.analytical.placer import AnalyticalPlacer
from submissions.core.eval import visualize

placer = AnalyticalPlacer(n_steps=0, legalize_iters=500, device=DEVICE)
pos = positions.clone().to(DEVICE).requires_grad_(True)
sizes_dev = benchmark.macro_sizes.to(DEVICE, dtype=torch.float32)
fixed_dev = benchmark.macro_fixed.to(DEVICE)
pos = placer._clamp_to_canvas(pos, sizes_dev, benchmark)
pos = placer._restore_fixed(pos, benchmark, fixed_dev)
pos = placer._legalize_push_apart(pos, sizes_dev, benchmark, fixed_dev)
pos = placer._restore_fixed(pos, benchmark, fixed_dev).detach().cpu()

result = tilos_evaluate(pos, benchmark, plc)
print(
    f"Proxy: {result['proxy_cost']:.4f}  "
    f"WL: {result['wirelength_cost']:.4f}  "
    f"Density: {result['density_cost']:.4f}  "
    f"Congestion: {result['congestion_cost']:.4f}  "
    f"Overlaps: {result['overlap_count']}"
)

final_path = CHECKPOINT_DIR / f"{benchmark.name}_v2_final.pt"
torch.save({"positions": pos, "tilos": result}, final_path)
visualize(pos, benchmark, plc=plc, save_path=str(CHECKPOINT_DIR / f"{benchmark.name}_v2_final.png"))
print(f"Saved final placement -> {final_path}")

## 9. Tuning notes

If the placement still collapses (everything in one corner):
- **Raise `w_density`** to 2500–5000
- Or **increase `n_bins`** to 12–16 (finer grid = stricter spreading)

If the placement is too spread out (looks like a grid that never collapsed):
- **Lower `w_density`** to 500–800
- Or **lower `n_bins`** to 6 (coarser grid)

If overlaps remain at the end:
- **Raise `w_overlap`** to 200
- Or **raise `legalize_iters`** in the evaluation cell

If wirelength is good but density cost in TILOS is high:
- Add a **slight ramp** on `w_density` (start lower, end higher) so wirelength
  can dominate early and density takes over late. Edit `train_v2` to anneal
  `w_density` like `gamma` does.